# Fase 3 — Regras de Associação: Candidatos 2026 (TSE)

## Pergunta de negócio

> **Que combinações de características dos candidatos aparecem juntas com mais frequência do que se fossem independentes? Em particular: quais atributos (partido, gênero, escolaridade, cor/raça, UF de nascimento e quartis das variáveis contínuas) estão associados a cada perfil identificado na Fase 2 — e, assim que a eleição ocorrer, a ser eleito?**

Até aqui, a clusterização (Fase 2) agrupou candidatos por **distância** em quatro variáveis numéricas (idade, tempo de estudo, patrimônio e despesas). Regras de associação fazem uma pergunta diferente: em vez de "quem está perto de quem", **quais combinações de atributos categóricos aparecem juntas com mais frequência do que o acaso explicaria**? A técnica clássica pra isso é o **Apriori**, que originalmente resolve o problema da "cesta de compras" (que produtos são comprados juntos) — aqui, cada candidato é uma cesta, e cada valor de cada variável categórica (partido, gênero, escolaridade...) é um item dessa cesta.


## Configuração

Use os **mesmos valores** que você usou nas Fases 0/1/2, para carregar o arquivo certo.


In [1]:
CARGO = "DEPUTADO FEDERAL"
UF = ['DF', 'GO', 'MS', 'MT']
ANO_ELEICAO = 2026
K_BISECTING = 5


## Preparando a base — reconstituindo os clusters nomeados da Fase 2

Apriori precisa de **itens categóricos**, não de distância — mas queremos incluir o cluster da Fase 2 como mais um item da cesta. Na Fase 2 (`02_clusterizacao.ipynb`), o **Bisecting K-Means com k=5** (critério de inércia média) foi o resultado final escolhido, e os cinco clusters foram batizados a partir da ficha técnica e da composição demográfica/partidária:

- **Patrimonializados, escolarizados e com campanha ativa**.
- **Campanha ativa sem patrimônio**.
- **Patrimonializados sem despesas declaradas**.
- **Jovens com baixa estrutura declarada**.
- **Escolarizados sem bens nem despesas**.

Recalculamos aqui, rapidamente, a mesma partição (mesma `SEMENTE`, mesmo critério, mesmos nomes) — mesma lógica de auto-suficiência já usada no `02_clusterizacao_outros.ipynb`.


In [2]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from mlxtend.frequent_patterns import apriori, association_rules
from pyvis.network import Network
from IPython.display import IFrame

sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)  # não truncar antecedents/consequents nas tabelas

SEMENTE = 42
np.random.seed(SEMENTE)  # mesmo racional de reprodutibilidade defensiva dos notebooks anteriores

nome_uf = str(UF) if UF is not None else 'BRASIL'
nome_cargo = CARGO.replace(' ', '_')
arquivo = f"dados/candidatos_{nome_cargo}_{nome_uf}_{ANO_ELEICAO}.csv"

df = pd.read_csv(arquivo)
print(f"Carregado: {arquivo}  ->  {df.shape[0]} candidatos, {df.shape[1]} colunas")
df.head()


Carregado: dados/candidatos_DEPUTADO_FEDERAL_['DF', 'GO', 'MS', 'MT']_2026.csv  ->  692 candidatos, 59 colunas


,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,SG_UF,SG_UE,NM_UE,CD_CARGO,DS_CARGO,SQ_CANDIDATO,NR_CANDIDATO,NM_CANDIDATO,NM_URNA_CANDIDATO,NM_SOCIAL_CANDIDATO,NR_CPF_CANDIDATO,DS_EMAIL,CD_SITUACAO_CANDIDATURA,DS_SITUACAO_CANDIDATURA,TP_AGREMIACAO,NR_PARTIDO,SG_PARTIDO,NM_PARTIDO,NR_FEDERACAO,NM_FEDERACAO,SG_FEDERACAO,DS_COMPOSICAO_FEDERACAO,SQ_COLIGACAO,NM_COLIGACAO,DS_COMPOSICAO_COLIGACAO,SG_UF_NASCIMENTO,DT_NASCIMENTO,NR_TITULO_ELEITORAL_CANDIDATO,CD_GENERO,DS_GENERO,CD_GRAU_INSTRUCAO,DS_GRAU_INSTRUCAO,CD_ESTADO_CIVIL,DS_ESTADO_CIVIL,CD_COR_RACA,DS_COR_RACA,CD_OCUPACAO,DS_OCUPACAO,CD_SIT_TOT_TURNO,DS_SIT_TOT_TURNO,BENS IMÓVEIS,BENS MÓVEIS,DINHEIRO,INVESTIMENTOS RENDA FIXA,INVESTIMENTOS RENDA VARIÁVEL,OUTROS BENS,Total_Bens,Total_Bens_Log,IDADE
0,08/09/2026,16:30:10,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,DF,DF,DISTRITO FEDERAL,6,DEPUTADO FEDERAL,70002532001,3022,ALESSANDRA ALMEIDA CHIANELLI DUTRA,ALE CHIANELLI,#NULO,58442472134,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,70001800084,PARTIDO ISOLADO,NOVO,DF,1972-11-28,11236642070,4,FEMININO,8,SUPERIOR COMPLETO,9,DIVORCIADO(A),1,BRANCA,171,JORNALISTA E REDATOR,-1,#NULO,0.0,130000.0,0.0,0.0,0.0,0.0,130000.0,11.775297,53
1,08/09/2026,16:30:10,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,DF,DF,DISTRITO FEDERAL,6,DEPUTADO FEDERAL,70002532002,3012,PAULO CESAR ECHEBARRIA DE CARVALHO,PAULO ECHEBARRIA,#NULO,37959077134,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,70001800084,PARTIDO ISOLADO,NOVO,GO,1965-10-20,7033782089,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),1,BRANCA,171,JORNALISTA E REDATOR,-1,#NULO,0.0,380000.0,0.0,0.0,0.0,15000000.0,15380000.0,16.548579,60
2,08/09/2026,16:30:10,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,DF,DF,DISTRITO FEDERAL,6,DEPUTADO FEDERAL,70002532003,3004,FLAVIO HENRIQUE TORMINN FLEURY,FLAVIO FLEURY,#NULO,88847560144,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,70001800084,PARTIDO ISOLADO,NOVO,GO,1981-09-01,46935651074,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),1,BRANCA,114,FISIOTERAPEUTA E TERAPEUTA OCUPACIONAL,-1,#NULO,0.0,53000.0,0.0,0.0,0.0,0.0,53000.0,10.878066,45
3,08/09/2026,16:30:10,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,DF,DF,DISTRITO FEDERAL,6,DEPUTADO FEDERAL,70002532004,3030,CARINE TEIXEIRA MARQUES,CARINE TEIXEIRA,#NULO,1642767182,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,70001800084,PARTIDO ISOLADO,NOVO,GO,1986-08-25,52572671066,4,FEMININO,8,SUPERIOR COMPLETO,3,CASADO(A),2,PRETA,257,EMPRESÁRIO,-1,#NULO,0.0,0.0,0.0,0.0,120000.0,0.0,120000.0,11.695255,40
4,08/09/2026,16:30:10,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,DF,DF,DISTRITO FEDERAL,6,DEPUTADO FEDERAL,70002532005,3010,EMILIO KERBER FILHO,EMILIO KERBER,#NULO,92759130053,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,70001800084,PARTIDO ISOLADO,NOVO,PR,1976-08-31,263044410167,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),1,BRANCA,185,ESCRITOR E CRÍTICO,-1,#NULO,1380000.0,0.0,0.0,0.0,0.0,0.0,1380000.0,14.137595,50


In [3]:
ANOS_ESTUDO_POR_GRAU = {
    'ANALFABETO': 0,
    'LÊ E ESCREVE': 1,
    'ENSINO FUNDAMENTAL INCOMPLETO': 4,
    'ENSINO FUNDAMENTAL COMPLETO': 8,
    'ENSINO MÉDIO INCOMPLETO': 9,
    'ENSINO MÉDIO COMPLETO': 11,
    'SUPERIOR INCOMPLETO': 13,
    'SUPERIOR COMPLETO': 16,
    # 'NÃO DIVULGÁVEL' fica de fora de propósito — mesma decisão da Versão 2 do notebook de clusterização
}

df['ANOS_ESTUDO'] = df['DS_GRAU_INSTRUCAO'].map(ANOS_ESTUDO_POR_GRAU)

# Mesma integração de despesas usada nas Fases 1 e 2.
arquivos_despesas = [
    Path(f'dados/despesas_contratadas_candidatos_{ANO_ELEICAO}_{uf}.csv')
    for uf in UF
]
arquivos_ausentes = [str(p) for p in arquivos_despesas if not p.exists()]
if arquivos_ausentes:
    raise FileNotFoundError('Arquivos de despesas ausentes: ' + ', '.join(arquivos_ausentes))

partes_despesas = []
for caminho in arquivos_despesas:
    parte = pd.read_csv(
        caminho, sep=';', encoding='latin-1',
        usecols=['DS_CARGO', 'SQ_CANDIDATO', 'VR_DESPESA_CONTRATADA'],
        dtype={'SQ_CANDIDATO': 'int64'}
    )
    parte = parte[parte['DS_CARGO'].str.upper() == CARGO.upper()].copy()
    parte['VR_DESPESA_CONTRATADA'] = pd.to_numeric(
        parte['VR_DESPESA_CONTRATADA'].astype(str)
        .str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce'
    )
    partes_despesas.append(parte)

gasto_por_candidato = (
    pd.concat(partes_despesas, ignore_index=True)
    .groupby('SQ_CANDIDATO', as_index=False)['VR_DESPESA_CONTRATADA'].sum(min_count=1)
    .rename(columns={'VR_DESPESA_CONTRATADA': 'Total_Gasto_Campanha'})
)
df = df.merge(gasto_por_candidato, on='SQ_CANDIDATO', how='left')
df['Total_Gasto_Campanha'] = df['Total_Gasto_Campanha'].fillna(0)
df['Total_Gasto_Campanha_Log'] = np.log1p(df['Total_Gasto_Campanha'])


In [4]:
colunas_driver = ['IDADE', 'ANOS_ESTUDO', 'Total_Bens_Log', 'Total_Gasto_Campanha_Log']

Q1, Q3 = df['Total_Bens_Log'].quantile([0.25, 0.75])
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

df_cluster = df[(df['Total_Bens_Log'] <= limite_superior) & (df['ANOS_ESTUDO'].notna())].copy()
print(f"Base para clusterização/regras: {df_cluster.shape[0]} candidatos "
      f"(descartados {df.shape[0] - df_cluster.shape[0]} por patrimônio extremo ou escolaridade não divulgada)")

scaler = MinMaxScaler()
X = scaler.fit_transform(df_cluster[colunas_driver])


Base para clusterização/regras: 692 candidatos (descartados 0 por patrimônio extremo ou escolaridade não divulgada)


In [5]:
def inercia(indices):
    if len(indices) < 2:
        return 0.0
    pontos = X[indices]
    centro = pontos.mean(axis=0)
    return float(((pontos - centro) ** 2).sum(axis=1).mean())


def bisecting_kmeans_ate_k(X, k_max, random_state=SEMENTE):
    """Mesma lógica do 02_clusterizacao.ipynb (Parte 2, critério de inércia média) — mas só
    devolve a partição final em k_max clusters, sem guardar o histórico de divisões (essa
    exploração já foi feita e documentada lá)."""
    clusters = {0: np.arange(X.shape[0])}
    proximo_id = 1
    for _ in range(1, k_max):
        id_escolhido = max(clusters, key=lambda cid: inercia(clusters[cid]))
        indices_pai = clusters[id_escolhido]
        if len(indices_pai) < 2:
            break
        km2 = KMeans(n_clusters=2, random_state=random_state, n_init=10)
        labels2 = km2.fit_predict(X[indices_pai])
        id_a, id_b = proximo_id, proximo_id + 1
        proximo_id += 2
        del clusters[id_escolhido]
        clusters[id_a] = indices_pai[labels2 == 0]
        clusters[id_b] = indices_pai[labels2 == 1]
    return clusters


k_bisecting = K_BISECTING

clusters_finais = bisecting_kmeans_ate_k(X, k_bisecting)
ids_por_tamanho = sorted(clusters_finais, key=lambda cid: len(clusters_finais[cid]), reverse=True)

rotulos = np.empty(X.shape[0], dtype=object)
for letra, cid in zip([chr(65 + i) for i in range(len(ids_por_tamanho))], ids_por_tamanho):
    rotulos[clusters_finais[cid]] = letra
df_cluster['cluster_bisecting'] = rotulos

# mesmos nomes definidos no 02_clusterizacao.ipynb, a partir da mesma ficha técnica
NOMES_CLUSTER_BISECTING = {
    'A': 'Patrimonializados, escolarizados e com campanha ativa',
    'B': 'Campanha ativa sem patrimônio',
    'C': 'Patrimonializados sem despesas declaradas',
    'D': 'Jovens com baixa estrutura declarada',
    'E': 'Escolarizados sem bens nem despesas',
}
df_cluster['cluster'] = df_cluster['cluster_bisecting'].map(NOMES_CLUSTER_BISECTING)

df_cluster['cluster'].value_counts()


cluster
Patrimonializados, escolarizados e com campanha ativa    418
Campanha ativa sem patrimônio                            111
Patrimonializados sem despesas declaradas                 78
Jovens com baixa estrutura declarada                      44
Escolarizados sem bens nem despesas                       41
Name: count, dtype: int64

## Montando a "cesta" de itens

Apriori não trabalha com números contínuos como o K-Means — ele precisa de **itens binários** (presente/ausente), do mesmo jeito que "pão" e "leite" são itens numa cesta de supermercado. Cada linha vira uma "cesta" (um candidato), e cada valor de cada variável categórica vira um "item" via one-hot encoding (`SG_PARTIDO=PT`, `DS_GENERO=FEMININO`, `cluster=Campanha ativa sem patrimônio`, ...).

Variáveis escolhidas:
- `SG_PARTIDO`, `DS_GENERO`, `DS_GRAU_INSTRUCAO`, `DS_ESTADO_CIVIL`, `DS_COR_RACA`, `SG_UF_NASCIMENTO` — perfil do candidato.
- `cluster` — o nome do cluster da Fase 2 (Bisecting K-Means, k=5), recalculado acima. Incluir o cluster como item deixa a pergunta "o que caracteriza cada cluster?" ser respondida de novo do ponto de vista de regras — e permite achar combinações de 2+ variáveis que sozinhas não bastariam.
- Quartis de `IDADE`, `ANOS_ESTUDO`, `Total_Bens` e `Total_Gasto_Campanha` — as quatro variáveis contínuas usadas na clusterização, transformadas em itens de Q1 (menores valores) a Q4 (maiores valores). Empates são distribuídos de forma determinística para manter quatro faixas.
- `SG_UF` — entra porque o recorte reúne quatro UFs; só ficaria de fora num recorte de UF única.
- `DS_SIT_TOT_TURNO` (eleito ou não) — só entra depois que a eleição de fato ocorrer (a coluna hoje vem toda com o mesmo valor, `#NULO`). A seção que usa isso fica pronta, mas roda condicionalmente lá na frente.

`DS_OCUPACAO` fica de fora por ora porque possui muitas categorias; várias ocupações teriam suporte baixo demais para regras confiáveis. Fica como exercício agrupá-las em categorias maiores (ex.: "política", "direito", "saúde", "educação", "empresário") e testar.


In [6]:
colunas_apriori = ['SG_PARTIDO', 'DS_GENERO', 'DS_GRAU_INSTRUCAO', 'DS_ESTADO_CIVIL',
                   'DS_COR_RACA', 'SG_UF_NASCIMENTO', 'cluster']

variaveis_quartis = {
    'IDADE': 'IDADE_QUARTIL',
    'ANOS_ESTUDO': 'ANOS_ESTUDO_QUARTIL',
    'Total_Bens': 'TOTAL_BENS_QUARTIL',
    'Total_Gasto_Campanha': 'TOTAL_DESPESAS_QUARTIL',
}
rotulos_quartis = ['Q1', 'Q2', 'Q3', 'Q4']
for coluna, coluna_quartil in variaveis_quartis.items():
    df_cluster[coluna_quartil] = pd.qcut(
        df_cluster[coluna].rank(method='first'), q=4, labels=rotulos_quartis
    )

colunas_apriori.extend(variaveis_quartis.values())

if UF is None or not isinstance(UF, str):
    colunas_apriori.append('SG_UF')

# a eleição de 2026 ainda não ocorreu -> DS_SIT_TOT_TURNO vem toda com o mesmo valor por enquanto
usar_resultado_eleicao = df_cluster['DS_SIT_TOT_TURNO'].nunique() > 1
if usar_resultado_eleicao:
    colunas_apriori.append('DS_SIT_TOT_TURNO')

cesta_base = df_cluster[['SQ_CANDIDATO'] + colunas_apriori].copy()
cesta = pd.get_dummies(cesta_base, columns=colunas_apriori, prefix_sep='=')
cesta = cesta.set_index('SQ_CANDIDATO')

print(f"{cesta.shape[0]} candidatos (cestas) x {cesta.shape[1]} itens possíveis (um por valor de cada variável)")
cesta_base.head()


692 candidatos (cestas) x 97 itens possíveis (um por valor de cada variável)


,SQ_CANDIDATO,SG_PARTIDO,DS_GENERO,DS_GRAU_INSTRUCAO,DS_ESTADO_CIVIL,DS_COR_RACA,SG_UF_NASCIMENTO,cluster,IDADE_QUARTIL,ANOS_ESTUDO_QUARTIL,TOTAL_BENS_QUARTIL,TOTAL_DESPESAS_QUARTIL,SG_UF
0,70002532001,NOVO,FEMININO,SUPERIOR COMPLETO,DIVORCIADO(A),BRANCA,DF,"Patrimonializados, escolarizados e com campanha ativa",Q3,Q2,Q2,Q3,DF
1,70002532002,NOVO,MASCULINO,SUPERIOR COMPLETO,CASADO(A),BRANCA,GO,"Patrimonializados, escolarizados e com campanha ativa",Q4,Q2,Q4,Q2,DF
2,70002532003,NOVO,MASCULINO,SUPERIOR COMPLETO,CASADO(A),BRANCA,GO,"Patrimonializados, escolarizados e com campanha ativa",Q2,Q2,Q2,Q2,DF
3,70002532004,NOVO,FEMININO,SUPERIOR COMPLETO,CASADO(A),PRETA,GO,"Patrimonializados, escolarizados e com campanha ativa",Q1,Q2,Q2,Q2,DF
4,70002532005,NOVO,MASCULINO,SUPERIOR COMPLETO,CASADO(A),BRANCA,PR,"Patrimonializados, escolarizados e com campanha ativa",Q3,Q2,Q4,Q2,DF


## Itens frequentes: o que aparece sozinho, com que frequência

O Apriori funciona em duas etapas. Primeiro, encontra **itemsets frequentes** — combinações de 1, 2, 3... itens que aparecem juntas em pelo menos `min_suporte` das cestas. Só depois, na segunda etapa, ele vira **regras** (`A → B`). Começamos pelos itemsets de um item só — a frequência "crua" de cada valor, antes de cruzar qualquer coisa.


In [7]:
min_suporte = 0.02  # ~14 candidatos; ajuste se quiser regras mais raras (menor) ou mais comuns (maior)

itens_frequentes = apriori(cesta, min_support=min_suporte, use_colnames=True, max_len=3)
print(f"{len(itens_frequentes)} itemsets frequentes (min_suporte={min_suporte})")

individuais = itens_frequentes[itens_frequentes['itemsets'].apply(lambda x: len(x) == 1)].copy()
individuais['item'] = individuais['itemsets'].apply(lambda x: next(iter(x)))
individuais['qtd_candidatos'] = (individuais['support'] * len(cesta)).round().astype(int)
individuais.sort_values('support', ascending=False)[['item', 'support', 'qtd_candidatos']].head(15)


3532 itemsets frequentes (min_suporte=0.02)


,item,support,qtd_candidatos
26,DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,0.651734,451
21,DS_GENERO=MASCULINO,0.611272,423
49,"cluster=Patrimonializados, escolarizados e com campanha ativa",0.604046,418
31,DS_COR_RACA=BRANCA,0.544798,377
28,DS_ESTADO_CIVIL=CASADO(A),0.518786,359
20,DS_GENERO=FEMININO,0.388728,269
67,SG_UF=GO,0.378613,262
33,DS_COR_RACA=PARDA,0.333815,231
30,DS_ESTADO_CIVIL=SOLTEIRO(A),0.315029,218
37,SG_UF_NASCIMENTO=GO,0.271676,188


### Suporte, confiança e lift — o que cada métrica mede

Regras de associação têm o formato **"se A, então B"** (A = antecedente, B = consequente), e três números resumem o quão forte é essa relação:

- **Suporte** — fração de candidatos em que A e B aparecem juntos: `suporte(A→B) = P(A e B)`. Mede o quão comum é a combinação; suporte baixo (poucos candidatos) é terreno fértil pra coincidência.
- **Confiança** — entre os candidatos que têm A, qual fração também tem B: `confiança(A→B) = P(B | A) = suporte(A e B) / suporte(A)`. É a "taxa de acerto" da regra.
- **Lift** — o quanto a confiança da regra é maior (ou menor) do que se A e B fossem independentes: `lift(A→B) = confiança(A→B) / suporte(B)`. `lift = 1` → A e B são independentes (a regra não diz nada); `lift > 1` → aparecem juntos mais do que o esperado ao acaso; `lift < 1` → aparecem juntos menos do que o esperado.

**Um cuidado importante nesta base:** são 692 candidatos distribuídos por muitos partidos e quatro UFs. Algumas categorias continuam pequenas, então poucas coincidências podem produzir lift alto sem representar um padrão estável. Por isso toda tabela de regras abaixo vem com uma coluna `qtd_candidatos` — sempre olhem ela antes de confiar num lift alto.


In [8]:
regras = association_rules(itens_frequentes, metric='lift', min_threshold=1)
regras['qtd_candidatos'] = (regras['support'] * len(cesta)).round().astype(int)
regras = regras.sort_values('confidence', ascending=False)


def formatar_regras(df_regras):
    """Troca antecedents/consequents (frozenset) por texto legível, só pra exibição —
    os dados originais (usados pra filtrar/ordenar) continuam intactos."""
    df_fmt = df_regras.copy()
    df_fmt['antecedents'] = df_fmt['antecedents'].apply(lambda x: ', '.join(sorted(str(i) for i in x)))
    df_fmt['consequents'] = df_fmt['consequents'].apply(lambda x: ', '.join(sorted(str(i) for i in x)))
    return df_fmt


print(f"{len(regras)} regras (lift >= 1)")
formatar_regras(regras[['antecedents', 'consequents', 'qtd_candidatos', 'support', 'confidence', 'lift']].head(10))


12080 regras (lift >= 1)


,antecedents,consequents,qtd_candidatos,support,confidence,lift
2514,"DS_GENERO=FEMININO, cluster=Jovens com baixa estrutura declarada",TOTAL_DESPESAS_QUARTIL=Q1,24,0.034682,1.0,4.000000
1295,"ANOS_ESTUDO_QUARTIL=Q4, SG_PARTIDO=PL",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,14,0.020231,1.0,1.534368
4549,"DS_COR_RACA=BRANCA, DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",ANOS_ESTUDO_QUARTIL=Q1,48,0.069364,1.0,4.000000
9656,"SG_UF_NASCIMENTO=GO, cluster=Patrimonializados sem despesas declaradas",SG_UF=GO,24,0.034682,1.0,2.641221
5310,"ANOS_ESTUDO_QUARTIL=Q4, DS_COR_RACA=PRETA",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,16,0.023121,1.0,1.534368
4510,"DS_ESTADO_CIVIL=SOLTEIRO(A), DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",ANOS_ESTUDO_QUARTIL=Q1,53,0.076590,1.0,4.000000
4577,"DS_COR_RACA=PARDA, DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",ANOS_ESTUDO_QUARTIL=Q1,42,0.060694,1.0,4.000000
5275,"ANOS_ESTUDO_QUARTIL=Q4, DS_COR_RACA=PARDA",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,51,0.073699,1.0,1.534368
5745,"ANOS_ESTUDO_QUARTIL=Q3, IDADE_QUARTIL=Q2",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,50,0.072254,1.0,1.534368
5719,"ANOS_ESTUDO_QUARTIL=Q4, IDADE_QUARTIL=Q1",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,29,0.041908,1.0,1.534368


### Cuidado: suporte pequeno infla o lift

Vamos ver isso na prática: as regras de maior lift da base inteira, sem nenhum filtro de suporte, comparadas às regras de maior lift **exigindo pelo menos 10 candidatos** por trás delas.


In [9]:
regras_1_consequente = regras[regras['consequents'].apply(lambda x: len(x) == 1)].copy()
colunas_exibir = ['antecedents', 'consequents', 'qtd_candidatos', 'confidence', 'lift']

print("--- Maior lift, sem filtro de suporte ---")
display(formatar_regras(regras_1_consequente.sort_values('lift', ascending=False)[colunas_exibir].head(5)))

print("--- Maior lift, exigindo qtd_candidatos >= 10 ---")
display(formatar_regras(regras_1_consequente[regras_1_consequente['qtd_candidatos'] >= 10]
        .sort_values('lift', ascending=False)[colunas_exibir].head(5)))


--- Maior lift, sem filtro de suporte ---


,antecedents,consequents,qtd_candidatos,confidence,lift
6326,"ANOS_ESTUDO_QUARTIL=Q2, SG_UF=GO",DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO,31,1.000000,9.611111
1141,"IDADE_QUARTIL=Q1, TOTAL_DESPESAS_QUARTIL=Q2",SG_PARTIDO=MISSÃO,20,0.416667,9.301075
10583,"TOTAL_BENS_QUARTIL=Q4, TOTAL_DESPESAS_QUARTIL=Q1",cluster=Patrimonializados sem despesas declaradas,18,0.947368,8.404858
10499,"ANOS_ESTUDO_QUARTIL=Q1, TOTAL_DESPESAS_QUARTIL=Q1",cluster=Jovens com baixa estrutura declarada,37,0.521127,8.195903
10469,"TOTAL_BENS_QUARTIL=Q1, TOTAL_DESPESAS_QUARTIL=Q1",cluster=Escolarizados sem bens nem despesas,36,0.480000,8.101463


--- Maior lift, exigindo qtd_candidatos >= 10 ---


,antecedents,consequents,qtd_candidatos,confidence,lift
6326,"ANOS_ESTUDO_QUARTIL=Q2, SG_UF=GO",DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO,31,1.000000,9.611111
1141,"IDADE_QUARTIL=Q1, TOTAL_DESPESAS_QUARTIL=Q2",SG_PARTIDO=MISSÃO,20,0.416667,9.301075
10583,"TOTAL_BENS_QUARTIL=Q4, TOTAL_DESPESAS_QUARTIL=Q1",cluster=Patrimonializados sem despesas declaradas,18,0.947368,8.404858
10499,"ANOS_ESTUDO_QUARTIL=Q1, TOTAL_DESPESAS_QUARTIL=Q1",cluster=Jovens com baixa estrutura declarada,37,0.521127,8.195903
10469,"TOTAL_BENS_QUARTIL=Q1, TOTAL_DESPESAS_QUARTIL=Q1",cluster=Escolarizados sem bens nem despesas,36,0.480000,8.101463


No primeiro grupo, categorias raras de partido, nascimento ou perfil podem produzir lift muito alto com poucos candidatos. No segundo grupo, com um piso de candidatos por trás da regra, sobram combinações mais estáveis e interessantes (ex.: quartis das variáveis contínuas e características cadastrais prevendo cluster). Essa é a leitura que vamos seguir usando daqui pra frente.

### Filtrando pra regras que valem a pena olhar


In [10]:
suporte_pratico = 0.03    # ~21 candidatos
confianca_pratica = 0.5

regras_simples = regras[
    regras['antecedents'].apply(lambda x: len(x) <= 2) & regras['consequents'].apply(lambda x: len(x) == 1)
]
regras_confiaveis = regras_simples[
    (regras_simples['support'] >= suporte_pratico) & (regras_simples['confidence'] >= confianca_pratica)
]

print(f"{len(regras_confiaveis)} regras com suporte >= {suporte_pratico} e confiança >= {confianca_pratica}")
formatar_regras(regras_confiaveis.sort_values('confidence', ascending=False)[colunas_exibir].head(15))


1777 regras com suporte >= 0.03 e confiança >= 0.5


,antecedents,consequents,qtd_candidatos,confidence,lift
2514,"DS_GENERO=FEMININO, cluster=Jovens com baixa estrutura declarada",TOTAL_DESPESAS_QUARTIL=Q1,24,1.0,4.000000
4549,"DS_COR_RACA=BRANCA, DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",ANOS_ESTUDO_QUARTIL=Q1,48,1.0,4.000000
9656,"SG_UF_NASCIMENTO=GO, cluster=Patrimonializados sem despesas declaradas",SG_UF=GO,24,1.0,2.641221
4510,"DS_ESTADO_CIVIL=SOLTEIRO(A), DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",ANOS_ESTUDO_QUARTIL=Q1,53,1.0,4.000000
4577,"DS_COR_RACA=PARDA, DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",ANOS_ESTUDO_QUARTIL=Q1,42,1.0,4.000000
5275,"ANOS_ESTUDO_QUARTIL=Q4, DS_COR_RACA=PARDA",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,51,1.0,1.534368
5745,"ANOS_ESTUDO_QUARTIL=Q3, IDADE_QUARTIL=Q2",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,50,1.0,1.534368
5719,"ANOS_ESTUDO_QUARTIL=Q4, IDADE_QUARTIL=Q1",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,29,1.0,1.534368
395,ANOS_ESTUDO_QUARTIL=Q4,DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,173,1.0,1.534368
393,ANOS_ESTUDO_QUARTIL=Q3,DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,173,1.0,1.534368


## O que prediz o cluster de um candidato?

Filtramos as regras cujo **consequente é um único item de cluster** — ou seja, "se o candidato tem tais características, ele cai no cluster X". É a mesma pergunta que a Fase 2 respondeu via ficha técnica, agora vista pelo ângulo de combinações de atributos categóricos.

Um cuidado extra aqui: os cinco clusters do Bisecting K-Means são desbalanceados — o maior reúne 418 candidatos, enquanto os dois menores têm 44 e 41. Ordenar apenas por confiança faria o grupo majoritário dominar a tabela; usar um piso alto e fixo poderia esconder os grupos menores. Por isso pegamos as regras de **maior lift dentro de cada cluster**, separadamente — assim os cinco aparecem e podemos comparar a força relativa de cada um.


In [11]:
def regras_para_consequente(regras_completas, prefixo, n_por_valor=6, min_suporte=min_suporte, max_antecedentes=2):
    """
    Entre as regras cujo único item consequente comece com `prefixo` (ex.: 'cluster='),
    devolve as `n_por_valor` de MAIOR LIFT pra CADA valor distinto do consequente — sem piso
    de confiança fixo, porque um valor raro (poucos candidatos) não sustenta regras de
    confiança alta mesmo quando o lift é forte; ordenar só por confiança deixaria os valores
    mais comuns dominando a tabela inteira (ver célula anterior).
    """
    candidatas = regras_completas[
        regras_completas['consequents'].apply(lambda x: len(x) == 1 and next(iter(x)).startswith(prefixo))
        & regras_completas['antecedents'].apply(lambda x: len(x) <= max_antecedentes)
        & (regras_completas['support'] >= min_suporte)
    ].copy()
    candidatas['alvo'] = candidatas['consequents'].apply(lambda x: next(iter(x)))
    resultado = (candidatas.sort_values('lift', ascending=False)
                            .groupby('alvo', group_keys=False)
                            .head(n_por_valor)
                            .sort_values(['alvo', 'lift'], ascending=[True, False]))
    return resultado.drop(columns='alvo')


regras_cluster = regras_para_consequente(regras, 'cluster=')
print(f"{len(regras_cluster)} regras (até 6 por cluster, ordenadas por lift dentro de cada um)")
formatar_regras(regras_cluster[colunas_exibir])


30 regras (até 6 por cluster, ordenadas por lift dentro de cada um)


,antecedents,consequents,qtd_candidatos,confidence,lift
10406,"TOTAL_BENS_QUARTIL=Q1, TOTAL_DESPESAS_QUARTIL=Q2",cluster=Campanha ativa sem patrimônio,61,1.000000,6.234234
10413,"TOTAL_BENS_QUARTIL=Q1, TOTAL_DESPESAS_QUARTIL=Q3",cluster=Campanha ativa sem patrimônio,34,1.000000,6.234234
1593,"SG_PARTIDO=PSOL, TOTAL_BENS_QUARTIL=Q1",cluster=Campanha ativa sem patrimônio,14,0.875000,5.454955
10402,"ANOS_ESTUDO_QUARTIL=Q3, TOTAL_BENS_QUARTIL=Q1",cluster=Campanha ativa sem patrimônio,16,0.800000,4.987387
7355,"DS_ESTADO_CIVIL=DIVORCIADO(A), TOTAL_BENS_QUARTIL=Q1",cluster=Campanha ativa sem patrimônio,23,0.741935,4.625400
6680,"DS_ESTADO_CIVIL=CASADO(A), TOTAL_BENS_QUARTIL=Q1",cluster=Campanha ativa sem patrimônio,41,0.683333,4.260060
10469,"TOTAL_BENS_QUARTIL=Q1, TOTAL_DESPESAS_QUARTIL=Q1",cluster=Escolarizados sem bens nem despesas,36,0.480000,8.101463
10463,"ANOS_ESTUDO_QUARTIL=Q4, TOTAL_DESPESAS_QUARTIL=Q1",cluster=Escolarizados sem bens nem despesas,16,0.470588,7.942611
5596,"DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, TOTAL_DESPESAS_QUARTIL=Q1",cluster=Escolarizados sem bens nem despesas,28,0.358974,6.058787
10457,"ANOS_ESTUDO_QUARTIL=Q2, TOTAL_DESPESAS_QUARTIL=Q1",cluster=Escolarizados sem bens nem despesas,15,0.357143,6.027875


### Visualizando como grafo

Uma tabela com dezenas de regras é difícil de escanear — um grafo interativo ajuda a ver de relance quais itens aparecem mais como "explicação" (antecedentes) e quais aparecem mais como "resultado" (consequentes). Cada item vira um nó (`SG_PARTIDO=PCO`, `cluster=D`, ...); cada regra vira uma seta do antecedente pro consequente, com espessura proporcional à confiança. Os nós que começam com `cluster=` ficam destacados em vermelho e maiores, pra serem fáceis de achar no meio dos outros itens. Passe o mouse sobre uma seta pra ver confiança, lift e quantos candidatos sustentam aquela regra — e arraste os nós se a rede ficar embaralhada.


In [12]:
def grafo_regras_html(regras_plot, arquivo_html, titulo, prefixo_destaque='cluster='):
    """
    Desenha as regras como uma rede interativa (pyvis) e grava em `arquivo_html`. Cada REGRA
    vira um nó de antecedente e um nó de consequente (quando o antecedente tem mais de um
    item, eles aparecem juntos no mesmo nó, separados por "+") ligados por UMA única aresta
    — nunca uma aresta por item isolado, pra não sugerir que um item sozinho sustenta a
    confiança/lift de uma regra que na verdade depende da combinação inteira (era isso que
    causava setas duplicadas entre o mesmo par de nós). Nós que contêm algum item começando
    com `prefixo_destaque` (por padrão, os clusters da Fase 2) ficam destacados em vermelho e
    maiores. Espessura da aresta = confiança da regra; passe o mouse pra ver confiança, lift
    e quantos candidatos a sustentam. Os controles de física (embaixo do grafo) deixam
    ajustar a repulsão/gravidade entre os nós ao vivo.

    Retorna um IFrame carregando o arquivo — se aparecer em branco/preto aqui embaixo (alguns
    ambientes bloqueiam o JavaScript de saídas de célula em notebooks "não confiáveis"), abra
    `arquivo_html` direto no navegador; o grafo é o mesmo.
    """
    print(titulo)
    net = Network(height='600px', width='100%', bgcolor='#222222', font_color='white',
                  notebook=True, directed=True, cdn_resources='in_line')

    nos_adicionados = set()
    for _, row in regras_plot.iterrows():
        antecedente = ' + '.join(sorted(str(item) for item in row['antecedents']))
        consequente = ' + '.join(sorted(str(item) for item in row['consequents']))

        for label, itemset in [(antecedente, row['antecedents']), (consequente, row['consequents'])]:
            if label in nos_adicionados:
                continue
            destaque = any(str(item).startswith(prefixo_destaque) for item in itemset)
            net.add_node(
                label, label, title=label,
                color='#e41a1c' if destaque else '#1f78b4',
                size=30 if destaque else 12,
                font={'size': 22 if destaque else 14},
            )
            nos_adicionados.add(label)

        titulo_aresta = (f"Confiança: {row['confidence']:.2f} | Lift: {row['lift']:.2f} | "
                          f"{row['qtd_candidatos']} candidatos")
        net.add_edge(antecedente, consequente, value=row['confidence'], title=titulo_aresta)

    net.show_buttons(filter_=['physics'])
    net.save_graph(arquivo_html)
    print(f"Grafo salvo em: {arquivo_html} — se não aparecer abaixo, abra esse arquivo no navegador.")
    return IFrame(arquivo_html, width='100%', height=750)


grafo_regras_html(regras_cluster.head(20), 'grafo_regras_cluster.html', 'O que prediz o cluster de um candidato?')


O que prediz o cluster de um candidato?
Grafo salvo em: grafo_regras_cluster.html — se não aparecer abaixo, abra esse arquivo no navegador.


## O que caracteriza quem já é de um cluster?

A seção anterior fixou o **consequente** (`cluster=X`) e perguntou o que o prediz. Agora invertemos: fixamos o **antecedente** — "dado que o candidato é do cluster X, ..." — e olhamos pra que outros itens aparecem como consequente com lift alto. É uma pergunta diferente, não a mesma coisa ao contrário: uma regra `A → B` de lift alto não garante que `B → A` também tenha lift alto (confiança e suporte de cada lado são diferentes), então vale olhar as duas direções separadamente.


In [13]:
def regras_para_antecedente(regras_completas, item_fixo, n=8, min_suporte=min_suporte, max_consequentes=1):
    """
    Entre as regras cujo antecedente é EXATAMENTE {item_fixo} (um único item), devolve as
    `n` de maior lift. Direção oposta de `regras_para_consequente`: lá fixamos o "efeito"
    (o consequente) e procurávamos o que o prediz; aqui fixamos a "causa" (o antecedente) e
    olhamos o que ela prediz.
    """
    candidatas = regras_completas[
        regras_completas['antecedents'].apply(lambda x: len(x) == 1 and next(iter(x)) == item_fixo)
        & regras_completas['consequents'].apply(lambda x: len(x) <= max_consequentes)
        & (regras_completas['support'] >= min_suporte)
    ]
    return candidatas.sort_values('lift', ascending=False).head(n)


itens_cluster = [f'cluster={nome}' for nome in NOMES_CLUSTER_BISECTING.values()]
regras_por_cluster_fixo = {
    item: regras_para_antecedente(regras, item)
    for item in itens_cluster
}

for item_fixo, subset in regras_por_cluster_fixo.items():
    print(f"--- {item_fixo} ({len(subset)} regras) ---")
    display(formatar_regras(subset[colunas_exibir]))


--- cluster=Patrimonializados, escolarizados e com campanha ativa (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
787,"cluster=Patrimonializados, escolarizados e com campanha ativa",TOTAL_DESPESAS_QUARTIL=Q4,170,0.406699,1.626794
111,"cluster=Patrimonializados, escolarizados e com campanha ativa",SG_PARTIDO=PL,42,0.100478,1.545136
203,"cluster=Patrimonializados, escolarizados e com campanha ativa",SG_PARTIDO=PT,28,0.066986,1.545136
239,"cluster=Patrimonializados, escolarizados e com campanha ativa",SG_PARTIDO=UNIÃO,21,0.050239,1.511546
783,"cluster=Patrimonializados, escolarizados e com campanha ativa",TOTAL_BENS_QUARTIL=Q4,155,0.370813,1.483254
781,"cluster=Patrimonializados, escolarizados e com campanha ativa",TOTAL_BENS_QUARTIL=Q3,153,0.366029,1.464115
167,"cluster=Patrimonializados, escolarizados e com campanha ativa",SG_PARTIDO=PSD,25,0.059809,1.379585
219,"cluster=Patrimonializados, escolarizados e com campanha ativa",SG_PARTIDO=REPUBLICANOS,38,0.090909,1.338491


--- cluster=Campanha ativa sem patrimônio (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
721,cluster=Campanha ativa sem patrimônio,TOTAL_BENS_QUARTIL=Q1,102,0.918919,3.675676
192,cluster=Campanha ativa sem patrimônio,SG_PARTIDO=PSOL,14,0.126126,2.909309
86,cluster=Campanha ativa sem patrimônio,SG_PARTIDO=PDT,18,0.162162,2.439483
723,cluster=Campanha ativa sem patrimônio,TOTAL_DESPESAS_QUARTIL=Q2,67,0.603604,2.414414
586,cluster=Campanha ativa sem patrimônio,DS_COR_RACA=PRETA,20,0.180180,1.833598
411,cluster=Campanha ativa sem patrimônio,DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO,20,0.180180,1.731732
715,cluster=Campanha ativa sem patrimônio,IDADE_QUARTIL=Q1,44,0.396396,1.585586
717,cluster=Campanha ativa sem patrimônio,ANOS_ESTUDO_QUARTIL=Q1,44,0.396396,1.585586


--- cluster=Patrimonializados sem despesas declaradas (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
763,cluster=Patrimonializados sem despesas declaradas,TOTAL_DESPESAS_QUARTIL=Q1,76,0.974359,3.897436
19,cluster=Patrimonializados sem despesas declaradas,SG_PARTIDO=DC,16,0.205128,3.301133
759,cluster=Patrimonializados sem despesas declaradas,TOTAL_BENS_QUARTIL=Q2,40,0.512821,2.051282
755,cluster=Patrimonializados sem despesas declaradas,ANOS_ESTUDO_QUARTIL=Q1,26,0.333333,1.333333
753,cluster=Patrimonializados sem despesas declaradas,IDADE_QUARTIL=Q4,24,0.307692,1.230769
503,cluster=Patrimonializados sem despesas declaradas,DS_ESTADO_CIVIL=SOLTEIRO(A),30,0.384615,1.220889
766,cluster=Patrimonializados sem despesas declaradas,SG_UF=MT,18,0.230769,1.157191
631,cluster=Patrimonializados sem despesas declaradas,SG_UF_NASCIMENTO=GO,24,0.307692,1.132570


--- cluster=Jovens com baixa estrutura declarada (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
748,cluster=Jovens com baixa estrutura declarada,TOTAL_DESPESAS_QUARTIL=Q1,44,1.000000,4.000000
745,cluster=Jovens com baixa estrutura declarada,ANOS_ESTUDO_QUARTIL=Q1,37,0.840909,3.363636
747,cluster=Jovens com baixa estrutura declarada,TOTAL_BENS_QUARTIL=Q1,35,0.795455,3.181818
340,cluster=Jovens com baixa estrutura declarada,DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO,19,0.431818,2.692056
743,cluster=Jovens com baixa estrutura declarada,IDADE_QUARTIL=Q1,27,0.613636,2.454545
500,cluster=Jovens com baixa estrutura declarada,DS_ESTADO_CIVIL=SOLTEIRO(A),34,0.772727,2.452877
262,cluster=Jovens com baixa estrutura declarada,DS_GENERO=FEMININO,24,0.545455,1.403177
559,cluster=Jovens com baixa estrutura declarada,DS_COR_RACA=PARDA,20,0.454545,1.361669


--- cluster=Escolarizados sem bens nem despesas (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
740,cluster=Escolarizados sem bens nem despesas,TOTAL_DESPESAS_QUARTIL=Q1,41,1.000000,4.000000
739,cluster=Escolarizados sem bens nem despesas,TOTAL_BENS_QUARTIL=Q1,36,0.878049,3.512195
736,cluster=Escolarizados sem bens nem despesas,ANOS_ESTUDO_QUARTIL=Q4,16,0.390244,1.560976
735,cluster=Escolarizados sem bens nem despesas,ANOS_ESTUDO_QUARTIL=Q2,15,0.365854,1.463415
733,cluster=Escolarizados sem bens nem despesas,IDADE_QUARTIL=Q3,14,0.341463,1.365854
531,cluster=Escolarizados sem bens nem despesas,DS_COR_RACA=BRANCA,28,0.682927,1.253542
298,cluster=Escolarizados sem bens nem despesas,DS_GENERO=MASCULINO,31,0.756098,1.236926
499,cluster=Escolarizados sem bens nem despesas,DS_ESTADO_CIVIL=SOLTEIRO(A),15,0.365854,1.161334


### Visualizando como grafo (antecedente fixo)

Mesmo grafo interativo de antes, só que agora os cinco clusters são o ponto de partida fixo, e os itens que aparecem como consequência é que variam.


In [14]:
regras_antecedente_fixo = pd.concat(regras_por_cluster_fixo.values())
grafo_regras_html(regras_antecedente_fixo, 'grafo_regras_antecedente_fixo.html', 'O que caracteriza quem já é de um cluster?')


O que caracteriza quem já é de um cluster?
Grafo salvo em: grafo_regras_antecedente_fixo.html — se não aparecer abaixo, abra esse arquivo no navegador.


## E quando a eleição já tiver ocorrido?

Esta seção só roda depois que a Fase 0 for rodada de novo com o resultado oficial (`DS_SIT_TOT_TURNO` deixa de vir toda com o mesmo valor `#NULO`) — por enquanto, ela se anuncia e não faz nada. Repare que não precisamos escrever nenhum código novo: como `DS_SIT_TOT_TURNO` já entrou na cesta lá na preparação (quando aplicável), a mesma `regras_para_consequente` e o mesmo `grafo_regras_html` servem pra essa pergunta também.


In [15]:
if usar_resultado_eleicao:
    regras_eleicao = regras_para_consequente(regras, 'DS_SIT_TOT_TURNO=')
    print(f"{len(regras_eleicao)} regras apontando para o resultado da eleição")
    display(formatar_regras(regras_eleicao[colunas_exibir].head(15)))
    grafo_regras_html(regras_eleicao.head(20), 'grafo_regras_eleicao.html', 'O que prediz ser eleito?', prefixo_destaque='DS_SIT_TOT_TURNO=')
else:
    print("DS_SIT_TOT_TURNO ainda não tem informação real (a eleição de 2026 não ocorreu) — "
          "essa seção fica pronta pra usar assim que a Fase 0 for rodada de novo com o resultado oficial. "
          "Por enquanto, pulamos.")


DS_SIT_TOT_TURNO ainda não tem informação real (a eleição de 2026 não ocorreu) — essa seção fica pronta pra usar assim que a Fase 0 for rodada de novo com o resultado oficial. Por enquanto, pulamos.


---
## Fechamento e próximos passos

Resumindo o roteiro: transformamos cada candidato numa "cesta" de atributos categóricos, quartis das quatro variáveis contínuas e o cluster da Fase 2; mineramos itemsets frequentes com o Apriori, geramos regras `A → B` com suporte/confiança/lift e analisamos obrigatoriamente as duas direções: **Cluster → características** e **características → Cluster**.

**Exercícios sugeridos:**
- Agrupar `DS_OCUPACAO` em categorias maiores e incluir na cesta.
- Variar `min_suporte` (célula da seção "Itens frequentes") e ver como o número de regras muda.
- Replicar para `SENADOR` ou `DEPUTADO FEDERAL`, na sua UF — só muda a célula de configuração, do mesmo jeito que nas fases anteriores.
- Depois que a eleição ocorrer: rodar a Fase 0 de novo, e conferir se a seção "E quando a eleição já tiver ocorrido?" acima encontra regras fortes ligadas a `DS_SIT_TOT_TURNO=ELEITO`.

Próxima etapa do pipeline: `04_deteccao_anomalias.ipynb`.
